# 11 - Single Intent End-to-End

## Objetivo
Percorrer o ciclo completo de um único intent através de toda a
arquitetura real:

```text
Intent -> Validation -> Planning -> ExecutionPlan -> Deployment
       -> Reservation -> Simulation -> IntentObservation -> Assurance -> SATISFIED
```

usando o cenário declarativo real `scenarios/linear_three_nodes.yaml`
(o mesmo arquivo usado no notebook 10), sem nenhuma etapa simulada ou
pré-calculada fora do SeQUeNCe.

## Conceitos
- `experiments.scenarios.Scenario.load` resolve o YAML do cenário e do
  intent num `EntanglementIntent` real.
- `IntentPlanner.plan` produz o HOW (`ExecutionPlan`) a partir do WHAT
  (`EntanglementIntent`) - nunca o contrário.
- A rota que o `ExecutionPlan` escolheu é apenas uma **decisão do
  planejador**; a prova de que o SeQUeNCe realmente aceitou essa rota é o
  campo `path` do objeto `Reservation` real (`RSVPProtocol.accepted_reservations`),
  verificado explicitamente abaixo.
- `assurance.evaluator.evaluate_intent` julga o intent usando apenas
  evidência real (`EventTypes.DELIVERY`), nunca a estimativa do planner.


## Configuração

Carregando o cenário e construindo o plano (ainda sem tocar o SeQUeNCe).


In [1]:
from sequence.constants import SECOND
from ibqn.experiments.scenarios import Scenario
from ibqn.network.capabilities import NetworkCapabilities
from ibqn.planning.planner import IntentPlanner

scenario = Scenario.load("../scenarios/linear_three_nodes.yaml")
intent = scenario.intents[0]
topology_spec = scenario.topology_spec()
SEED = scenario.spec.simulation.seed

capabilities = NetworkCapabilities(topology_spec)
planner = IntentPlanner(capabilities)
plan = planner.plan(intent)

print(f"intent: {intent.id} ({intent.endpoints.source} -> {intent.endpoints.destination})")
print(f"plan feasible: {plan.feasible}, route: {' -> '.join(plan.route)}")


2026-10-02 15:03:40,308 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'requested_pairs' -> 'reserved_memory_slots' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:40,314 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'start_time' -> 'start_time_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:40,316 INFO     ibqn.ibqn.intent.models intent_id=- - IntentRequirements: migrating legacy field 'duration' -> 'duration_s' (see docs/intent_resource_semantics.md)


2026-10-02 15:03:40,320 INFO     ibqn.ibqn.planning.planner intent_id=intent-001 - planned route A->R1->B, estimated_fidelity=0.7300, purification=False


intent: intent-001 (A -> B)
plan feasible: True, route: A -> R1 -> B


## Execução

Deploy real: aplica a rota, submete a reserva ao `NetworkManager`, roda a `Timeline`.


In [2]:
from ibqn.network.sequence_adapter import SequenceAdapter
from ibqn.execution.sequence_executor import SequenceExecutor
from ibqn.intent.repository import IntentRepository

adapter = SequenceAdapter(topology_spec, seed=SEED)
repository = IntentRepository()
executor = SequenceExecutor(adapter, repository)

executor.deploy(intent, plan)
executor.run()

repository.get(intent.id).lifecycle.status


2026-10-02 15:03:40,348 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - topology built: 3 routers, 2 quantum links, formalism=bell_diagonal, seed=42


2026-10-02 15:03:40,351 INFO     ibqn.ibqn.intent.repository intent_id=intent-001 - intent registered


2026-10-02 15:03:40,352 INFO     ibqn.ibqn.intent.repository intent_id=intent-001 - intent transitioned to VALIDATED (schema validated)


2026-10-02 15:03:40,354 INFO     ibqn.ibqn.intent.repository intent_id=intent-001 - intent transitioned to PLANNING (invoking IntentPlanner)


2026-10-02 15:03:40,355 INFO     ibqn.ibqn.intent.repository intent_id=intent-001 - intent transitioned to PLANNED (selected by ShortestHopCountRouting; hop fidelities [0.85, 0.85])


2026-10-02 15:03:40,356 INFO     ibqn.ibqn.intent.repository intent_id=intent-001 - intent transitioned to DEPLOYING (applying route A->R1->B and submitting reservation to NetworkManager (purification_mode=until_target))


2026-10-02 15:03:40,358 INFO     ibqn.ibqn.execution.sequence_executor intent_id=intent-001 - submitting reservation A -> B, reserved_memory_slots=10, min_fidelity=0.650, purification_mode=until_target


2026-10-02 15:03:40,359 INFO     ibqn.ibqn.network.sequence_adapter intent_id=- - running simulation, stop_time=0.100s


2026-10-02 15:03:40,362 INFO     ibqn.ibqn.intent.repository intent_id=intent-001 - intent transitioned to ACTIVE (reservation approved)


<IntentStatus.ACTIVE: 'ACTIVE'>

### Confirmando que a rota observada corresponde à rota do plano

Lendo o objeto `Reservation` real que o SeQUeNCe negociou (não o plano).


In [3]:
source_router = adapter.get_router(intent.endpoints.source)
rsvp = source_router.network_manager.protocol_stack[-1]
reservation = rsvp.accepted_reservations[0]

print(f"plan.route              = {plan.route}")
print(f"reservation.path (real) = {reservation.path}")
assert reservation.path == plan.route, "the network must have actually used the route the planner chose"
print("Confirmed: the accepted SeQUeNCe reservation used exactly the planned route.")


plan.route              = ['A', 'R1', 'B']
reservation.path (real) = ['A', 'R1', 'B']
Confirmed: the accepted SeQUeNCe reservation used exactly the planned route.


## Assurance

Avaliação das condições declaradas no intent, usando exclusivamente evidência real (`DELIVERY`).


In [4]:
from ibqn.assurance.evaluator import evaluate_intent
from ibqn.assurance.telemetry import collect_intent_evidence
from ibqn.intent.models import IntentStatus

evidence = collect_intent_evidence(intent)
evaluation = evaluate_intent(intent, evidence)
final_status = IntentStatus.SATISFIED if evaluation.satisfied else IntentStatus.VIOLATED
repository.transition(intent.id, final_status, "all success conditions met" if evaluation.satisfied else "; ".join(evaluation.violations), sim_time=adapter.get_timeline().now() / SECOND)

repository.get(intent.id).lifecycle.status


2026-10-02 15:03:43,470 INFO     ibqn.ibqn.intent.repository intent_id=intent-001 - intent transitioned to SATISFIED (all success conditions met)


<IntentStatus.SATISFIED: 'SATISFIED'>

## Resultados


In [5]:
from ibqn.demos.tables import evaluation_table

print(f"delivered pairs: {len(evidence.delivered_pairs)}")
print(f"average fidelity (observed): {sum(p.fidelity for p in evidence.delivered_pairs) / len(evidence.delivered_pairs):.4f}")
print(f"minimum fidelity (observed): {min(p.fidelity for p in evidence.delivered_pairs):.4f}")
print(f"throughput (observed): {len(evidence.delivered_pairs) / intent.requirements.duration_s:.2f} pairs/s")
evaluation_table(evaluation)

delivered pairs: 335
average fidelity (observed): 0.7288
minimum fidelity (observed): 0.7265
throughput (observed): 6700.00 pairs/s


,metric,operator,expected,observed,passed,evidence_source
0,delivered_pairs,>=,10.00,335.00000,True,335 DELIVERY event(s) tagged intent_id='intent...
1,average_fidelity,>=,0.65,0.72884,True,335 DELIVERY event(s) tagged intent_id='intent...


### Intent (WHAT) vs. ExecutionPlan (HOW)


In [6]:
import pandas as pd

comparison_df = pd.DataFrame([
    {"Intent - WHAT": f"source={intent.endpoints.source}, destination={intent.endpoints.destination}",
     "ExecutionPlan - HOW": f"route={' -> '.join(plan.route)}"},
    {"Intent - WHAT": f"min_fidelity={intent.requirements.min_fidelity}",
     "ExecutionPlan - HOW": f"estimated_fidelity={plan.estimated_metrics.fidelity:.4f}"},
    {"Intent - WHAT": f"reserved_memory_slots={intent.requirements.reserved_memory_slots}",
     "ExecutionPlan - HOW": f"reservations={[(r.node_id, r.memories_required) for r in plan.reservations]}"},
    {"Intent - WHAT": f"start_time_s={intent.requirements.start_time_s}s, duration_s={intent.requirements.duration_s}s",
     "ExecutionPlan - HOW": f"reservation window (real): [{reservation.start_time / SECOND:.3f}, {reservation.end_time / SECOND:.3f}) s"},
    {"Intent - WHAT": f"validation.success_conditions={[str(c) for c in intent.validation.success_conditions]}",
     "ExecutionPlan - HOW": "evaluated post-hoc by assurance.evaluator against DELIVERY evidence - not a planning-time field"},
    {"Intent - WHAT": f"policy.allow_purification={intent.policy.allow_purification}, policy.allow_rerouting={intent.policy.allow_rerouting}",
     "ExecutionPlan - HOW": f"requires_purification={plan.requires_purification}; {plan.swapping_strategy_note}"},
])
comparison_df

,Intent - WHAT,ExecutionPlan - HOW
0,"source=A, destination=B",route=A -> R1 -> B
1,min_fidelity=0.65,estimated_fidelity=0.7300
2,reserved_memory_slots=10,"reservations=[('A', 10), ('R1', 20), ('B', 10)]"
3,"start_time_s=0.01s, duration_s=0.05s","reservation window (real): [0.010, 0.060) s"
4,validation.success_conditions=['delivered_pair...,evaluated post-hoc by assurance.evaluator agai...
5,"policy.allow_purification=True, policy.allow_r...",requires_purification=False; swap order follow...


In [7]:
from ibqn.demos.tables import lifecycle_table

lifecycle_table(repository.get(intent.id).lifecycle)


,from_status,to_status,reason,sim_time_s
0,(none),RECEIVED,intent received,0.000000
1,RECEIVED,VALIDATED,schema validated,0.000000
2,VALIDATED,PLANNING,invoking IntentPlanner,0.000000
3,PLANNING,PLANNED,selected by ShortestHopCountRouting; hop fidel...,0.000000
4,PLANNED,DEPLOYING,applying route A->R1->B and submitting reserva...,0.000000
5,DEPLOYING,ACTIVE,reservation approved,0.000400
6,ACTIVE,SATISFIED,all success conditions met,0.060164


In [8]:
assert repository.get(intent.id).lifecycle.status == IntentStatus.SATISFIED
assert evaluation.satisfied is True
assert all(0 <= p.fidelity <= 1 for p in evidence.delivered_pairs)
assert isinstance(len(evidence.delivered_pairs), int) and len(evidence.delivered_pairs) >= 0
print("End-to-end cycle confirmed: intent reached SATISFIED with all conditions passing.")


End-to-end cycle confirmed: intent reached SATISFIED with all conditions passing.


## Interpretação
A rota que o `IntentPlanner` escolheu antes de qualquer simulação
(`plan.route`) é idêntica ao caminho que a reserva real do SeQUeNCe
efetivamente usou (`reservation.path`) - a arquitetura nunca aplica uma
rota diferente da que planejou. Todas as métricas em "Resultados" vêm de
`assurance.telemetry.collect_intent_evidence`, ou seja, de eventos
`DELIVERY` reais tagueados com `intent_id`, nunca de
`intent.requirements`/`plan.estimated_metrics`.

## Limitações
- Este cenário não aciona purificação (`requires_purification=False`);
  o ciclo com purificação é coberto no notebook 17.
- `reserved_memory_slots` funciona como tamanho do pool de memórias, não
  como teto de entrega - o número de pares entregues pode (e geralmente
  excede) o valor solicitado, dado tempo suficiente
  (`docs/sequence_integration.md`, `docs/intent_resource_semantics.md`).

## Conclusão
Ciclo completo demonstrado: intent declarativo -> validação -> planejamento
-> execução real -> observação -> assurance -> `SATISFIED`, com a rota
planejada confirmada byte a byte contra a reserva real aceita pelo
SeQUeNCe.